In [8]:
from mmedit.apis import init_model
print("BasicVSR++ is ready!")


BasicVSR++ is ready!


In [9]:
import os
import cv2
import torch
import mmcv
import shutil
import numpy as np
from mmedit.apis import init_model, restoration_video_inference
from IPython.display import Video, display


In [10]:
# Paths
config_path = "/home/patrick/BasicVSR_PlusPlus/configs/basicvsr_plusplus_reds4.py"
checkpoint_path = "checkpoints/basicvsr_plusplus_c64n7_8x1_600k_reds4_20210217-db622b2f.pth"

print("Config and checkpoint ready!")

Config and checkpoint ready!


In [11]:
print(torch.cuda.is_available())

True


In [13]:
# ============================================================
# FOLDER-BACKED, CHUNK-BASED BASICVSR++ PIPELINE (OOM SAFE)
# ============================================================

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
input_video = "input.mp4"        # <-- your video filename
output_video = "output_vsr.mp4"  # final upscaled result

device = "cuda" if torch.cuda.is_available() else "cpu"
frames_dir = "lq/video1"
output_frames_dir = "vsr_frames"

CHUNK = 12     # frames per inference call
OVERLAP = 2    # temporal context (must be >= 1)

# ------------------------------------------------------------
# UTILS
# ------------------------------------------------------------
def reset_folder(folder):
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

reset_folder(frames_dir)
reset_folder(output_frames_dir)

# ============================================================
# STEP 1 — Extract frames from video
# ============================================================

cap = cv2.VideoCapture(input_video)
index = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imwrite(f"{frames_dir}/{index:08d}.png", frame)
    index += 1

fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"Extracted {index} frames into {frames_dir}")

# ============================================================
# STEP 2 — Load BasicVSR++ model
# ============================================================

print("Loading BasicVSR++ model...")
model = init_model(config_path, checkpoint_path, device=device)
model.eval()
print("Model loaded on:", device)

# ============================================================
# STEP 3 — Run super-resolution inference (CHUNKED, SAFE)
# ============================================================

@torch.no_grad()
def run_model_on_frames(frames_bgr):
    """
    frames_bgr: list of np.uint8 BGR frames
    returns: list of upscaled BGR frames
    """
    tensors = []
    for f in frames_bgr:
        rgb = (f[:, :, ::-1] / 255.0).astype("float32")
        t = torch.from_numpy(rgb).permute(2, 0, 1)
        tensors.append(t)

    inp = torch.stack(tensors).unsqueeze(0).to(device)  # (1,T,3,H,W)
    out = model(inp, test_mode=True)["output"]          # (1,T,3,H',W')
    out = out.squeeze(0).cpu().numpy()

    results = []
    for i in range(out.shape[0]):
        rgb = out[i].transpose(1, 2, 0)
        bgr = (rgb * 255).clip(0, 255).astype("uint8")[:, :, ::-1]
        results.append(bgr)

    # aggressive cleanup to keep VRAM flat
    del inp, out, tensors
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    return results


print("Running VSR inference (chunk-based)...")

frame_files = sorted(os.listdir(frames_dir))
buffer = []
written = 0

for fname in frame_files:
    frame = cv2.imread(os.path.join(frames_dir, fname))
    buffer.append(frame)

    # once buffer reaches CHUNK + OVERLAP, process safely
    if len(buffer) == CHUNK + OVERLAP:
        process_frames = buffer[:-OVERLAP]
        out_frames = run_model_on_frames(process_frames)

        for f in out_frames:
            cv2.imwrite(f"{output_frames_dir}/{written:08d}.png", f)
            written += 1

        buffer = buffer[-OVERLAP:]

# process remaining frames at EOF
if len(buffer) > 0:
    out_frames = run_model_on_frames(buffer)
    for f in out_frames:
        cv2.imwrite(f"{output_frames_dir}/{written:08d}.png", f)
        written += 1

print(f"Saved {written} upscaled frames into {output_frames_dir}")

# ============================================================
# STEP 4 — Reassemble output video
# ============================================================

print("Reassembling upscaled video...")

frame_files = sorted(os.listdir(output_frames_dir))
if len(frame_files) == 0:
    raise RuntimeError("ERROR: No output frames were generated!")

first = cv2.imread(os.path.join(output_frames_dir, frame_files[0]))
h, w, _ = first.shape

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (w, h))

for f in frame_files:
    frame = cv2.imread(os.path.join(output_frames_dir, f))
    out.write(frame)

out.release()
print("Saved upscaled video to:", output_video)

# ============================================================
# STEP 5 — Display result inline (optional)
# ============================================================

print("Showing result:")
display(Video(output_video, width=512))


2025-12-12 22:38:55,942 - mmedit - INFO - load checkpoint from http path: https://download.openmmlab.com/mmediting/restorers/basicvsr/spynet_20210409-c6c1bd09.pth


Extracted 97 frames into lq/video1
Loading BasicVSR++ model...
load checkpoint from local path: checkpoints/basicvsr_plusplus_c64n7_8x1_600k_reds4_20210217-db622b2f.pth
Model loaded on: cuda
Running VSR inference (chunk-based)...
Saved 97 upscaled frames into vsr_frames
Reassembling upscaled video...
Saved upscaled video to: output_vsr.mp4
Showing result:


In [ ]:
# This version does not store frames in folders. There may be a way to incorporate colorization and 
#   degradation removal here for a folderless solution.

"""
input_video = "input.mp4"       # <-- your video filename
output_video = "output_vsr.mp4" # final upscaled result
device = "cuda" if torch.cuda.is_available() else "cpu"

##############################################
# STEP 2 — Load BasicVSR++ model
##############################################

print("Loading BasicVSR++ model...")
model = init_model(config_path, checkpoint_path, device=device)
print("Model loaded on:", device)

################
##############################################
# STEP 3 — STREAMING VSR (Safe for 3060 Ti)
##############################################

print("Running VSR in streaming mode...")

def run_model_on_buffer(frames, model, device="cuda"):
    #Run BasicVSR++ on a small list of BGR frames.
    # Convert small buffer (T frames) → tensor: (1,T,3,H,W)
    tensors = []
    for f in frames:
        rgb = (f[:, :, ::-1] / 255.0).astype("float32")
        t = torch.from_numpy(rgb).permute(2,0,1)   # 3,H,W
        tensors.append(t)

    inp = torch.stack(tensors, dim=0).unsqueeze(0).to(device)  # 1,T,3,H,W

    with torch.no_grad():
        out = model(inp, test_mode=True)["output"]   # (1,T,3,H',W')

    out = out.squeeze(0).cpu().numpy()               # (T,3,H',W')

    # Convert back to list of BGR numpy frames
    result_frames = []
    for i in range(out.shape[0]):
        rgb = out[i].transpose(1,2,0)
        bgr = (rgb * 255).clip(0,255).astype("uint8")[:, :, ::-1]
        result_frames.append(bgr)

    # Clean GPU
    del inp, out, tensors
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    return result_frames


# -------- STREAMING SETTINGS --------
CHUNK = 12       # frames processed per model call
OVERLAP = 2      # temporal context
video_in = input_video
video_out = output_video

cap = cv2.VideoCapture(video_in)
if not cap.isOpened():
    raise RuntimeError("Cannot open input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video FPS: {fps}, total frames: {total_frames}")

writer = None
buffer = []
written = 0

print("Starting streaming inference...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    buffer.append(frame)

    # once buffer reaches chunk+overlap, process all but the OVERLAP frames
    if len(buffer) == CHUNK + OVERLAP:

        process_frames = buffer[:-OVERLAP]    # keep last 2 frames for context
        out_frames = run_model_on_buffer(process_frames, model, device)

        # init writer once we know output resolution
        if writer is None:
            h, w, _ = out_frames[0].shape
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(video_out, fourcc, fps, (w,h))

        for f in out_frames:
            writer.write(f)
            written += 1

        # keep overlap frames for next chunk
        buffer = buffer[-OVERLAP:]

# process remaining frames after EOF
if len(buffer) > 0:
    out_frames = run_model_on_buffer(buffer, model, device)
    if writer is None:
        h, w, _ = out_frames[0].shape
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(video_out, fourcc, fps, (w,h))
    for f in out_frames:
        writer.write(f)
        written += 1

cap.release()
if writer:
    writer.release()

print("Streaming VSR complete.")
print("Total output frames written:", written)

##############################################
# STEP 5 — Display result inline (optional)
##############################################

print("Showing result:")
display(Video(output_video, width=512))
"""
